In [ ]:
import os
import time
import requests
import xml.etree.ElementTree as ET
import csv


## Harvest conicet records

In [ ]:

# Corrected OAI-PMH endpoint for CONICET
OAI_ENDPOINT = "https://ri.conicet.gov.ar/oai/request"
METADATA_PREFIX = "oai_dc"
OUTPUT_FOLDER = "conicet_records"

# OAI-PMH uses XML namespaces. These are required for ElementTree to find the tags.
NAMESPACES = {
    'oai': 'http://www.openarchives.org/OAI/2.0/',
    'oai_dc': 'http://www.openarchives.org/OAI/2.0/oai_dc/',
    'dc': 'http://purl.org/dc/elements/1.1/'
}

In [11]:


def harvest_conicet_records():
    # Create output folder if it doesn't exist
    if not os.path.exists(OUTPUT_FOLDER):
        os.makedirs(OUTPUT_FOLDER)

    # Initial request parameters
    params = {
        'verb': 'ListRecords',
        'metadataPrefix': METADATA_PREFIX
    }
    
    total_downloaded = 0
    first_record_printed = False
    
    print(f"Starting harvest from: {OAI_ENDPOINT}")
    
    while True:
        try:
            response = requests.get(OAI_ENDPOINT, params=params, timeout=30)
            response.raise_for_status()
        except requests.exceptions.RequestException as e:
            print(f"\nNetwork error: {e}. Retrying in 5 seconds...")
            time.sleep(5)
            continue
            
        # Parse the XML response
        root = ET.fromstring(response.content)
        
        # Check for OAI-PMH specific API errors
        error_node = root.find('oai:error', NAMESPACES)
        if error_node is not None:
            print(f"\nOAI Error {error_node.attrib.get('code', '')}: {error_node.text}")
            break
            
        # Extract all records from the current batch
        records = root.findall('.//oai:record', NAMESPACES)
        
        for record in records:
            header = record.find('oai:header', NAMESPACES)
            if header is None:
                continue
                
            identifier_node = header.find('oai:identifier', NAMESPACES)
            if identifier_node is not None:
                # Sanitize the identifier for safe use as a filename
                safe_id = identifier_node.text.replace(':', '_').replace('/', '_')
                filepath = os.path.join(OUTPUT_FOLDER, f"{safe_id}.xml")
                
                # Convert the record XML tree back to a string
                record_xml = ET.tostring(record, encoding='unicode')
                
                # Save individual record to disk
                with open(filepath, 'w', encoding='utf-8') as f:
                    f.write(record_xml)
                
                total_downloaded += 1
                
                # 1. Print the first full record exactly once to verify
                if not first_record_printed:
                    print("\n" + "="*50)
                    print("FIRST RECORD PREVIEW:")
                    print("="*50)
                    print(record_xml)
                    print("="*50 + "\n")
                    first_record_printed = True
                
                # 2. Continuous counting on the same terminal line
                print(f"\rRecords downloaded: {total_downloaded}", end="", flush=True)
                
        # 3. Handle Pagination via ResumptionTokens
        # OAI-PMH only returns a few hundred records at a time. The resumptionToken gets the next batch.
        resumption_token = root.find('.//oai:resumptionToken', NAMESPACES)
        
        if resumption_token is not None and resumption_token.text:
            # When using a resumptionToken, it replaces all other parameters (like metadataPrefix)
            params = {
                'verb': 'ListRecords',
                'resumptionToken': resumption_token.text
            }
            # Brief pause to be polite to the CONICET servers
            time.sleep(1)
        else:
            print("\n\nHarvest complete. No more records to fetch.")
            break

if __name__ == "__main__":
    harvest_conicet_records()

Starting harvest from: https://ri.conicet.gov.ar/oai/request

FIRST RECORD PREVIEW:
<ns0:record xmlns:dc="http://purl.org/dc/elements/1.1/" xmlns:ns0="http://www.openarchives.org/OAI/2.0/" xmlns:ns1="http://www.openarchives.org/OAI/2.0/oai_dc/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"><ns0:header><ns0:identifier>oai:ri.conicet.gov.ar:11336/179477</ns0:identifier><ns0:datestamp>2024-01-12T04:51:20Z</ns0:datestamp><ns0:setSpec>com_11336_35</ns0:setSpec><ns0:setSpec>com_11336_14</ns0:setSpec><ns0:setSpec>col_11336_36</ns0:setSpec><ns0:setSpec>snrd</ns0:setSpec></ns0:header><ns0:metadata><ns1:dc xsi:schemaLocation="http://www.openarchives.org/OAI/2.0/oai_dc/ http://www.openarchives.org/OAI/2.0/oai_dc.xsd">
<dc:identifier>http://hdl.handle.net/11336/179477</dc:identifier>
<dc:title>Paisajes mesetarios en Patagonia: Tecnología de Pampa del asador - Lago Guitarra (Santa Cruz)</dc:title>
<dc:creator>Cassiodoro, Gisela Eva</dc:creator>
<dc:subject>MESETAS</dc:subject>
<dc:subject>T

ParseError: no element found: line 1, column 359 (<string>)

## Unify xml into one csv

In [ ]:

INPUT_FOLDER = "conicet_records"
OUTPUT_CSV = "conicet_dataset.csv"

# Standard Dublin Core fields + our custom 'filiation' + the OAI identifier
FIELD_NAMES = [
    "oai_identifier", "title", "creator", "subject", 
    "description", "filiation", "date", "type", 
    "identifier", "language", "rights", "format", 
    "relation", "publisher", "source", "contributor", "coverage"
]

def process_records_to_csv():
    # Check if the folder exists
    if not os.path.exists(INPUT_FOLDER):
        print(f"Error: The folder '{INPUT_FOLDER}' does not exist.")
        return

    # Get a list of all XML files
    files = [f for f in os.listdir(INPUT_FOLDER) if f.endswith('.xml')]
    total_files = len(files)
    
    if total_files == 0:
        print("No XML files found in the folder.")
        return
        
    print(f"Found {total_files} XML files. Starting conversion to CSV...")

    # Open the CSV file for writing
    with open(OUTPUT_CSV, mode='w', newline='', encoding='utf-8') as csv_file:
        # We use DictWriter to easily map dictionary keys to CSV columns
        writer = csv.DictWriter(csv_file, fieldnames=FIELD_NAMES, extrasaction='ignore')
        writer.writeheader()

        for i, filename in enumerate(files, 1):
            filepath = os.path.join(INPUT_FOLDER, filename)
            
            try:
                tree = ET.parse(filepath)
                root = tree.getroot()
            except ET.ParseError:
                # If a file downloaded incompletely or is corrupt, skip it safely
                continue

            # Dictionary to hold lists of values for the current record
            row_data = {field: [] for field in FIELD_NAMES}

            # 1. Extract the primary OAI Identifier from the header
            oai_namespace = "http://www.openarchives.org/OAI/2.0/"
            header_id_node = root.find(f'.//{{{oai_namespace}}}identifier')
            if header_id_node is not None and header_id_node.text:
                row_data['oai_identifier'].append(header_id_node.text.strip())
            else:
                row_data['oai_identifier'].append(filename)

            # 2. Extract all Dublin Core metadata tags
            dc_namespace = "http://purl.org/dc/elements/1.1/"
            
            for elem in root.iter():
                # Check if the element belongs to the Dublin Core namespace
                if elem.tag.startswith(f"{{{dc_namespace}}}"):
                    # Strip the namespace to just get the tag name (e.g., 'title', 'creator')
                    tag_name = elem.tag.replace(f"{{{dc_namespace}}}", "")
                    
                    # Safely get text and clean whitespace
                    text = elem.text.strip() if elem.text else ""
                    if not text:
                        continue
                        
                    # 3. Apply custom logic for description vs filiation
                    if tag_name == "description":
                        if text.startswith("Fil.") or text.startswith("Fil:"):
                            row_data["filiation"].append(text)
                        else:
                            row_data["description"].append(text)
                            
                    # Add standard tags (title, creator, subject, etc.)
                    elif tag_name in row_data:
                        row_data[tag_name].append(text)
            
            # 4. Join multiple items in the same category with a pipe character
            final_row = {key: "|".join(values) for key, values in row_data.items()}
            
            # Write the row to the CSV
            writer.writerow(final_row)

            # Continuous counting on the same terminal line
            print(f"\rProcessed {i}/{total_files} records", end="", flush=True)

    print(f"\n\nDone! Data successfully saved to '{OUTPUT_CSV}'.")

if __name__ == "__main__":
    process_records_to_csv()

Found 167301 XML files. Starting conversion to CSV...
Processed 167301/167301 records

Done! Data successfully saved to 'conicet_dataset.csv'.
